In [13]:
from pathlib import Path
import random
import numpy as np
import torch

from flowrec.data.datamodule import AmazonReviewDataConfig, AmazonReviewDataModule
from flowrec.models.sasrec import SASRecConfig, SASRec
from flowrec.ft_methods.sft import SFTConfig, SFTMethod
from flowrec.training import TrainerConfig, Trainer

from flowrec.ft_methods.sft.metrics import item_ranking_metrics

In [14]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [15]:
@torch.no_grad()
def compute_test_hit_ndcg_coverage(
    model: torch.nn.Module,
    dm: AmazonReviewDataModule,
    *,
    ks=(5, 10, 20),
    pad_item_id: int = 0,
) -> dict[str, float]:
    """Compute NDCG@K, HitRate@K, Coverage@K on test data
    """
    device = next(model.parameters()).device
    model.eval()

    vocab_size = dm.item_vocab_size
    recommendable = max(vocab_size - 1, 1)

    # Accumulate instance-level hit and ndcg
    hits = {k: [] for k in ks}
    ndcg_sums = {k: 0.0 for k in ks}
    ndcg_counts = {k: 0 for k in ks}

    recommended_sets = {k: set() for k in ks}

    for batch in dm.test_dataloader():
        for key, val in batch.items():
            if torch.is_tensor(val):
                batch[key] = val.to(device, non_blocking=True)

        out = model(batch)
        scores = out.scores  # [B, V]
        target_item_ids = batch["target_item_ids"]

        targets = target_item_ids.long()
        valid = targets != pad_item_id
        if valid.sum() == 0:
            continue

        max_k = max(ks)
        topk_items = torch.topk(scores, k=max_k, dim=-1).indices  # [B, max_k]
        topk_valid = topk_items[valid]
        targets_valid = targets[valid]

        for k in ks:
            kk = int(k)
            topk_k = topk_valid[:, :kk]

            matches = topk_k.eq(targets_valid.unsqueeze(1))
            hit = matches.any(dim=1)
            hits[k].append(hit.float().cpu())

            rank_pos = matches.float().argmax(dim=1)
            ndcg = torch.zeros_like(hit, dtype=torch.float32)
            if hit.any():
                ndcg[hit] = 1.0 / torch.log2(rank_pos[hit].float() + 2.0)
            ndcg_sums[k] += float(ndcg.mean().item())
            ndcg_counts[k] += 1

            uniq = torch.unique(topk_k).detach().cpu().tolist()
            recommended_sets[k].update(int(x) for x in uniq if int(x) != pad_item_id)

    metrics: dict[str, float] = {}
    for k in ks:
        hit_t = torch.cat(hits[k], dim=0) if hits[k] else torch.zeros(0)
        hit_rate = float(hit_t.mean().item()) if hit_t.numel() else 0.0
        coverage = len(recommended_sets[k]) / recommendable
        ndcg = ndcg_sums[k] / max(ndcg_counts[k], 1)
        metrics[f"ndcg@{k}"] = ndcg
        metrics[f"hit_rate@{k}"] = hit_rate
        metrics[f"coverage@{k}"] = coverage

    return metrics

In [16]:
reviews_path = Path("../data/raw/reviews_Beauty.json.gz")
metadata_path = Path("../data/raw/meta_Beauty.json.gz")

data_cfg = AmazonReviewDataConfig(
    reviews_path=reviews_path,
    metadata_path=metadata_path,
    semantic_id_path=None,
    min_user_interactions=5,
    min_item_interactions=5,
    split_strategy="leave_one_out",
    max_history_len=50,
    history_padding="right",
    batch_size=256,
    eval_batch_size=512,
)

dm = AmazonReviewDataModule(data_cfg)
dm.prepare_data()
dm.setup()

print("Dataset summary:")
print(dm.summary())
print("Item vocab size:", dm.item_vocab_size)

Dataset summary:
{'num_users': 22363, 'num_items': 12101, 'num_train_examples': 131413, 'num_val_examples': 22363, 'num_test_examples': 22363, 'num_interactions': 198502, 'split_strategy': 'leave_one_out', 'global_val_cutoff': None, 'global_test_cutoff': None, 'sid_len': None, 'item_vocab_size': 12102}
Item vocab size: 12102


In [17]:
model_cfg = SASRecConfig(
    item_vocab_size=dm.item_vocab_size,
    max_history_len=data_cfg.max_history_len,
    d_model=256,
    nhead=8,
    num_layers=3,
    dropout=0.1,
)

sft_cfg = SFTConfig()
method_cfg = sft_cfg

trainer_cfg = TrainerConfig(
    max_epochs=10,
    gradient_accumulation_steps=4,
    use_amp=True,
    log_every_steps=50,
    validate_every_epochs=2,
    monitor="val/loss",
    early_stopping_patience=3,
    checkpoint_dir="../artifacts/sasrec_beauty",
)

In [18]:
seeds = [42, 43, 44]
ks = (5, 10, 20)
all_seed_metrics = []

for seed in seeds:
    print(f"\n===== Train/eval seed={seed} =====")
    set_seed(seed)

    model = SASRec(model_cfg)
    method = SFTMethod(method_cfg)

    trainer = Trainer.from_optimizer_config(
        model=model,
        method=method,
        cfg=trainer_cfg,
        lr=3e-4,
        weight_decay=1e-2,
    )

    _ = trainer.fit(dm.train_dataloader(), dm.val_dataloader())

    ckpt_path = f"../artifacts/sasrec_beauty/final_sasrec_seed_{seed}.pt"
    model.save_checkpoint(ckpt_path)

    test_metrics = compute_test_hit_ndcg_coverage(model, dm, ks=ks)
    test_metrics["seed"] = seed

    print("Test metrics:")
    for key in sorted([k for k in test_metrics.keys() if k != 'seed']):
        print(f"  {key}: {test_metrics[key]:.6f}")

    all_seed_metrics.append(test_metrics)


===== Train/eval seed=42 =====


/Users/maksimshuklin/miniconda3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
/Users/maksimshuklin/hse/flowrec/flowrec/training/trainer.py:183: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler(enabled=self._use_amp)


Test metrics:
  coverage@10: 0.779440
  coverage@20: 0.890835
  coverage@5: 0.636724
  hit_rate@10: 0.061262
  hit_rate@20: 0.081697
  hit_rate@5: 0.048115
  ndcg@10: 0.041994
  ndcg@20: 0.047245
  ndcg@5: 0.037585

===== Train/eval seed=43 =====
Test metrics:
  coverage@10: 0.785390
  coverage@20: 0.895546
  coverage@5: 0.640526
  hit_rate@10: 0.064079
  hit_rate@20: 0.085588
  hit_rate@5: 0.048473
  ndcg@10: 0.042722
  ndcg@20: 0.048283
  ndcg@5: 0.037567

===== Train/eval seed=44 =====
Test metrics:
  coverage@10: 0.776878
  coverage@20: 0.887117
  coverage@5: 0.637055
  hit_rate@10: 0.063766
  hit_rate@20: 0.083620
  hit_rate@5: 0.048205
  ndcg@10: 0.042809
  ndcg@20: 0.047971
  ndcg@5: 0.037660


In [19]:
df.to_csv("metrics_10_epochs.csv", index=False)

In [41]:
df

,ndcg@5,hit_rate@5,coverage@5,ndcg@10,hit_rate@10,coverage@10,ndcg@20,hit_rate@20,coverage@20,seed
0,0.037585,0.048115,0.636724,0.041994,0.061262,0.779440,0.047245,0.081697,0.890835,42
1,0.037567,0.048473,0.640526,0.042722,0.064079,0.785390,0.048283,0.085588,0.895546,43
2,0.037660,0.048205,0.637055,0.042809,0.063766,0.776878,0.047971,0.083620,0.887117,44


In [40]:
df_mean_std = pd.DataFrame({
    col: [f"{df[col].mean():.3f} ± {3 * df[col].std():.3f}"]
    for col in df.select_dtypes(include='number').columns
})
df_mean_std = df_mean_std.drop(columns=['seed'])
df_mean_std

,ndcg@5,hit_rate@5,coverage@5,ndcg@10,hit_rate@10,coverage@10,ndcg@20,hit_rate@20,coverage@20
0,0.038 ± 0.000,0.048 ± 0.001,0.638 ± 0.006,0.043 ± 0.001,0.063 ± 0.005,0.781 ± 0.013,0.048 ± 0.002,0.084 ± 0.006,0.891 ± 0.013


In [21]:
df.to_csv("data.csv", index=False)

In [23]:
trainer.state

TrainerState(epoch=9, global_step=5140, optimizer_step=1290, best_metric=8.62865609331186, best_epoch=7, bad_epochs=1, should_stop=False, history=[{'epoch': 0, 'global_step': 514, 'optimizer_step': 129, 'train': {'train/loss': 9.063735748396331, 'train/sft/base_loss': 9.063735748396331, 'train/sft/loss': 9.063735748396331, 'train/sft/ndcg@10': 0.007239332124223484, 'train/sft/ndcg@20': 0.009927829136447193, 'train/sft/ndcg@5': 0.005079559826607263, 'train/sft/recall@10': 0.015021344921262248, 'train/sft/recall@20': 0.025788924992341904, 'train/sft/recall@5': 0.008271632182657406}, 'last_checkpoint': '../artifacts/sasrec_beauty/last.pt', 'epoch_time_sec': 523.5170359611511}, {'epoch': 1, 'global_step': 1028, 'optimizer_step': 258, 'train': {'train/loss': 8.642774884587364, 'train/sft/base_loss': 8.642774884587364, 'train/sft/loss': 8.642774884587364, 'train/sft/ndcg@10': 0.027915200898243144, 'train/sft/ndcg@20': 0.03279075773066978, 'train/sft/ndcg@5': 0.02346684385731475, 'train/sft/r